In [0]:
from pyspark.sql import functions as F
SOURCE = "s3a://data5035-spring26/drone_data.json"

In [0]:
source_df = spark.read.json(SOURCE)
display(source_df.limit(10))

In [0]:
source_df.select(F.count_distinct("cesium_137_detector_calibration_timestamp")).show()

In [0]:
source_df.select(F.count_distinct("thorium_232_level")).show()

In [0]:
source_df.agg({'cesium_137_level': 'max'}).show()

In [0]:
source_df.agg({'cesium_137_level': 'min'}).show()

In [0]:
source_df.agg({'cesium_137_level': 'avg'}).show()

For the tables I would like to have - I think it would be really handy to have a table dedicated to the three t ypes of radiation/materials. Along with that, I think a tbale that focuses on what type of Detector Unit is being used would be helpful too. I think finding trends based on Detector Unit could be helpful in seeing if we have any hardware issues, and it could also help us see how calibration impacts the tool as well. 

In [0]:
#Creates the table as a dataframe
gamma_df = source_df.select(
    "COLLECTION_TIMESTAMP", "GPS_LAT", "GPS_LNG", "GPS_UNIT_NUMBER",
    "GPS_UNIT_CALIBRATION_TIMESTAMP", "GPS_UNIT_CALIBRATION_PRECISION",
    "GAMMA_LEVEL", "GAMMA_DETECTOR_UNIT_NUMBER",
    "GAMMA_DETECTOR_CALIBRATION_TIMESTAMP", "GAMMA_DETECTOR_CALIBRATION_PRECISION"
)

#you have to rewrite it as a table
gamma_df.write.mode("overwrite").saveAsTable("GAMMA")

In [0]:
source_df = spark.read.json(SOURCE)
display(gamma_df.limit(10)) #this is just to make sure I didn't mess anything up. 

I will copy this template in order to create more dimension tables. NExt will be Cesium and Thorum. Afterwards I will make a table change to impliment the 2 SCD format for extra credit. 

In [0]:

CESIUM_137_df = source_df.select(
    "COLLECTION_TIMESTAMP", "GPS_LAT", "GPS_LNG", "GPS_UNIT_NUMBER",
    "GPS_UNIT_CALIBRATION_TIMESTAMP", "GPS_UNIT_CALIBRATION_PRECISION",
    "CESIUM_137_LEVEL", "CESIUM_137_DETECTOR_UNIT_NUMBER",
    "CESIUM_137_DETECTOR_CALIBRATION_TIMESTAMP", "CESIUM_137_DETECTOR_CALIBRATION_PRECISION"
)


CESIUM_137_df.write.mode("overwrite").saveAsTable("CESIUM_137")

In [0]:
source_df = spark.read.json(SOURCE)
display(CESIUM_137_df.limit(10))

In [0]:
thorium_232_df = source_df.select(
    "COLLECTION_TIMESTAMP", "GPS_LAT", "GPS_LNG", "GPS_UNIT_NUMBER",
    "GPS_UNIT_CALIBRATION_TIMESTAMP", "GPS_UNIT_CALIBRATION_PRECISION",
    "thorium_232_LEVEL", "thorium_232_DETECTOR_UNIT_NUMBER",
    "thorium_232_DETECTOR_CALIBRATION_TIMESTAMP", "thorium_232_DETECTOR_CALIBRATION_PRECISION"
)




thorium_232_df.write.mode("overwrite").saveAsTable("thorium_232")

#this will be for my fact table where I hold the level of radiation for the detectors. 

In [0]:
datafacttable_df = source_df.select(
   "COLLECTION_TIMESTAMP", "GPS_LAT", "GPS_LNG", "GAMMA_LEVEL", "THORIUM_232_LEVEL", "CESIUM_137_LEVEL", "GPS_UNIT_NUMBER"
)

datafacttable_df.write.mode("overwrite").saveAsTable("datafacttable")

I made my fact table this data just because we just need event data. We don't need to many descriptive pieces. That is what the otehr tables are for. I will have my descriptors for the gama radiation, cesium radiation, and thalium radiation. After that I will make one for the equipment as additional description tables. 

In [0]:
#since we need to figure out the what - that means I need to make tables related to the technology being used. I will make a table for each device that is used, but also I am going to make a table that is a key to help things make sense. 

gammatools_df = source_df.select(
   "COLLECTION_TIMESTAMP", "GPS_UNIT_NUMBER", "GPS_UNIT_CALIBRATION_TIMESTAMP", "GPS_UNIT_CALIBRATION_PRECISION", "GAMMA_DETECTOR_UNIT_NUMBER", "GAMMA_DETECTOR_CALIBRATION_TIMESTAMP", "GAMMA_DETECTOR_CALIBRATION_PRECISION"
)
gammatools_df.write.mode("overwrite").saveAsTable("gammatools")

In [0]:
thorium_232_tools_df = source_df.select(
   "COLLECTION_TIMESTAMP", "GPS_UNIT_NUMBER", "GPS_UNIT_CALIBRATION_TIMESTAMP", "GPS_UNIT_CALIBRATION_PRECISION", "THORIUM_232_DETECTOR_UNIT_NUMBER", "THORIUM_232_DETECTOR_CALIBRATION_TIMESTAMP", "THORIUM_232_DETECTOR_CALIBRATION_PRECISION"
)
thorium_232_tools_df.write.mode("overwrite").saveAsTable("thorium_232_tools")

In [0]:
CESIUM_137_tools_df = source_df.select(
   "COLLECTION_TIMESTAMP", "GPS_UNIT_NUMBER", "GPS_UNIT_CALIBRATION_TIMESTAMP", "GPS_UNIT_CALIBRATION_PRECISION", "CESIUM_137_DETECTOR_UNIT_NUMBER", "CESIUM_137_DETECTOR_CALIBRATION_TIMESTAMP", "CESIUM_137_DETECTOR_CALIBRATION_PRECISION"
)
CESIUM_137_tools_df.write.mode("overwrite").saveAsTable("CESIUM_137_tools")
